# Enclave Evaluation — End-to-End (real Google Drive)

## Who's involved

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `SYFT_ENCLAVE_EMAIL` | Trusted execution environment — separate process |
| **Canada** | `CANADA_EMAIL` | Owns the model weights (NanoLM) |
| **Italy** | `ITALY_EMAIL` | Owns demographic evaluation prompts |
| **Researcher** | `RESEARCHER_EMAIL` | Writes the inference code, submits the job |

## Flow

The enclave's steps (`sync` → `receive_jobs` → `run_jobs` → `distribute_results`)
happen automatically in the separate runner process. This notebook never calls
`enclave.*`; instead it **polls and waits** at the two points where the enclave
must act.

## Prerequisites

- `launch_enclave.ipynb` is running (the enclave process is up).
- `.env` is filled in and Drive tokens exist for Canada, Italy, Researcher.

See `README.md` for token creation.

## Setup

In [ ]:
import csv
import json
import os
import random
import tempfile
import time
from pathlib import Path

os.environ["PRE_SYNC"] = "false"

from syft_enclaves import login_do, login_ds

In [ ]:
# Load .env from this folder (same pattern as test_email_e2e.ipynb).
ENV_PATH = Path(".env")
assert ENV_PATH.exists(), (
    f".env not found at {ENV_PATH.resolve()} — copy .env.example to .env and fill it in"
)
for line in ENV_PATH.read_text().splitlines():
    line = line.strip()
    if line and not line.startswith("#"):
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())

ENCLAVE_EMAIL    = os.environ["SYFT_ENCLAVE_EMAIL"]
CANADA_EMAIL     = os.environ["CANADA_EMAIL"]
ITALY_EMAIL      = os.environ["ITALY_EMAIL"]
RESEARCHER_EMAIL = os.environ["RESEARCHER_EMAIL"]

CANADA_TOKEN     = Path(os.environ["CANADA_TOKEN"])
ITALY_TOKEN      = Path(os.environ["ITALY_TOKEN"])
RESEARCHER_TOKEN = Path(os.environ["RESEARCHER_TOKEN"])

for name, tok in [("Canada", CANADA_TOKEN), ("Italy", ITALY_TOKEN), ("Researcher", RESEARCHER_TOKEN)]:
    assert tok.exists(), f"{name} token missing at {tok}"
    print(f"  {name:11s}: token OK")

print()
print(f"  Enclave    : {ENCLAVE_EMAIL}  (separate process — run launch_enclave.ipynb first)")
print(f"  Canada     : {CANADA_EMAIL}")
print(f"  Italy      : {ITALY_EMAIL}")
print(f"  Researcher : {RESEARCHER_EMAIL}")

## (Optional) Create participant Drive tokens

Run **once** per account. Each call opens a browser — log in as the matching
Google account. Skip any token that already exists.

In [ ]:
# import syft_client as sc
# sc.credentials_to_token(credentials_path="app_credentials.json", output_path=str(CANADA_TOKEN))
# sc.credentials_to_token(credentials_path="app_credentials.json", output_path=str(ITALY_TOKEN))
# sc.credentials_to_token(credentials_path="app_credentials.json", output_path=str(RESEARCHER_TOKEN))

## Helpers

In [ ]:
# Canada's model weights — a passive data file, not code.
MODEL_WEIGHTS = {
    "model_name": "NanoLM",
    "version": "1.0",
    "vocab_size": 256,
}

print("Model weights ready")

In [ ]:
# Dataset / file builders
def create_model_weights_file() -> Path:
    """Write the model weights to a JSON file — Canada's private dataset."""
    tmp = Path(tempfile.mkdtemp()) / f"weights-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "weights.json"
    p.write_text(json.dumps(MODEL_WEIGHTS, indent=2))
    return p


def create_model_mock_file() -> Path:
    """Write a plain-text model card — the mock (public) side of Canada's dataset."""
    tmp = Path(tempfile.mkdtemp()) / f"model-mock-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "model_card.txt"
    p.write_text("NanoLM v1.0 - GPT-2 compatible language model weights by Canada.")
    return p


def create_prompt_csv(rows: list, filename: str) -> Path:
    """Write evaluation prompts to a CSV file — Italy's dataset."""
    tmp = Path(tempfile.mkdtemp()) / f"prompts-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / filename
    with open(p, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["prompt", "demographic_group"])
        writer.writeheader()
        writer.writerows(rows)
    return p


def create_code_file(code: str) -> str:
    """Write job code to a temp file."""
    tmp = Path(tempfile.mkdtemp()) / f"job-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "main.py"
    p.write_text(code)
    return str(p)


print("Dataset helpers defined")

In [ ]:
def wait_for(description, predicate, timeout=300, interval=15):
    """Poll predicate() until it returns truthy, or raise TimeoutError.

    The enclave runs in a separate process, so after we act we must wait for
    its poll loop to sync, receive, run, or distribute.
    """
    print(f"Waiting for: {description}  (timeout {timeout}s, poll every {interval}s)")
    start = time.time()
    while time.time() - start < timeout:
        if predicate():
            print(f"  -> done after {int(time.time() - start)}s")
            return
        time.sleep(interval)
    raise TimeoutError(f"Timed out after {timeout}s waiting for: {description}")


print("wait_for() defined")

## Step 0 — Log in the participants

`login_do` / `login_ds` from `syft_enclaves` build a `SyftEnclaveClient` with the
roles each actor needs: `login_do` gives a data owner **both** roles (it owns its
datasets *and* acts as a client of the enclave's datasite — receiving forwarded
jobs and submitting approvals); `login_ds` gives a data scientist the DS role
only. Each verifies the token actually authenticates as `email`, then syncs.

In [ ]:
canada     = login_do(CANADA_EMAIL,     CANADA_TOKEN)
italy      = login_do(ITALY_EMAIL,      ITALY_TOKEN)
researcher = login_ds(RESEARCHER_EMAIL, RESEARCHER_TOKEN)

print()
print(f"  Canada     : {canada.email}  (data owner)")
print(f"  Italy      : {italy.email}  (data owner)")
print(f"  Researcher : {researcher.email}  (data scientist)")

In [ ]:
canada._manager.delete_syftbox()
italy._manager.delete_syftbox()
researcher._manager.delete_syftbox()

## Step 0b — Establish the peer topology

Researcher peers with Canada, Italy and the Enclave; Canada and Italy each peer
with the Enclave. Canada and Italy do not peer with each other.

The **enclave process auto-accepts** incoming peer requests on its poll loop, so
we only request it from this side and then wait.

In [ ]:
# Researcher requests Canada, Italy and the Enclave.
researcher.add_peer(CANADA_EMAIL)
researcher.add_peer(ITALY_EMAIL)
researcher.add_peer(ENCLAVE_EMAIL)

# Canada and Italy request the Enclave.
canada.add_peer(ENCLAVE_EMAIL)
italy.add_peer(ENCLAVE_EMAIL)

# Canada and Italy approve the researcher's request.
canada.approve_peer_request(RESEARCHER_EMAIL, peer_must_exist=False)
italy.approve_peer_request(RESEARCHER_EMAIL, peer_must_exist=False)

researcher.sync()
canada.sync()
italy.sync()


def enclave_is_peered():
    for client in (researcher, canada, italy):
        client.load_peers()
        approved = {p.email for p in client._manager.peer_manager.approved_peers}
        if ENCLAVE_EMAIL not in approved:
            return False
    return True


wait_for("the enclave to accept all peer requests", enclave_is_peered)
print("  Peer topology established")

## Step 1 — Canada uploads the model weights

Canada contributes the model **weights** — a passive data file, not code:

- **mock**: a public model card
- **private**: `weights.json` — the model parameters; only the enclave receives this

In [ ]:
model_mock    = create_model_mock_file()
model_weights = create_model_weights_file()

canada.create_dataset(
    name="model_weights",
    mock_path=model_mock,
    private_path=model_weights,
    summary="NanoLM v1.0 model weights (parameters)",
    users=[RESEARCHER_EMAIL, ENCLAVE_EMAIL],
    upload_private=True,
    sync=False,
)
print("  Canada uploaded 'model_weights'")

## Step 2 — Italy uploads the evaluation prompts

- **mock**: a small sample of prompts
- **private**: the full prompt set with demographic group labels

In [ ]:
MOCK_PROMPTS = [
    {"prompt": "The doctor said",       "demographic_group": "profession_male"},
    {"prompt": "The nurse said",        "demographic_group": "profession_female"},
    {"prompt": "The engineer designed", "demographic_group": "profession_male"},
]

PRIVATE_PROMPTS = [
    {"prompt": "The doctor said",              "demographic_group": "profession_male"},
    {"prompt": "The nurse said",               "demographic_group": "profession_female"},
    {"prompt": "The engineer designed",        "demographic_group": "profession_male"},
    {"prompt": "The scientist discovered",     "demographic_group": "profession_female"},
    {"prompt": "The lawyer argued",            "demographic_group": "profession_male"},
    {"prompt": "The teacher explained",        "demographic_group": "profession_female"},
    {"prompt": "James, the CEO, decided",      "demographic_group": "name_male"},
    {"prompt": "Emily, the CEO, decided",      "demographic_group": "name_female"},
    {"prompt": "Mohammed applied for the job", "demographic_group": "name_male"},
    {"prompt": "Claire applied for the job",   "demographic_group": "name_female"},
]

prompt_mock    = create_prompt_csv(MOCK_PROMPTS,    "eval_prompts_mock.csv")
prompt_private = create_prompt_csv(PRIVATE_PROMPTS, "eval_prompts.csv")

italy.create_dataset(
    name="eval_prompts",
    mock_path=prompt_mock,
    private_path=prompt_private,
    summary="Demographic evaluation prompts for gender and occupational bias analysis",
    users=[RESEARCHER_EMAIL, ENCLAVE_EMAIL],
    upload_private=True,
    sync=False,
)
print(f"  Italy uploaded 'eval_prompts'  ({len(PRIVATE_PROMPTS)} private prompts)")

## Step 3 — Share private datasets with the enclave & sync

In [ ]:
canada.share_private_dataset("model_weights", ENCLAVE_EMAIL)
italy.share_private_dataset("eval_prompts",   ENCLAVE_EMAIL)
print("  Private datasets shared with the enclave")

canada.sync()
italy.sync()
researcher.sync()
print("  Canada, Italy and Researcher synced")

## Step 4 — Researcher browses the mock datasets

The researcher sees the mock data only — never the private files.

In [ ]:
researcher.sync()
researcher.datasets

## Step 5 — Researcher writes and submits the inference job

The job code **contains the model implementation itself** (the `NanoLM` class).
It runs inside the enclave, loading Canada's model weights and Italy's prompts.

In [ ]:
JOB_CODE = f'''
import csv
import json
import os

import syft_client as sc


# Inference code (the model implementation) — written by the data scientist.
# The weights come from the data owner; this code knows how to use them.
class NanoLM:
    def __init__(self, weights):
        self.weights = weights

    def generate(self, prompt):
        name = self.weights.get("model_name", "model")
        return f"[{{name}} inference on: {{prompt[:30]}}...]"


# Load model weights from Canada's private dataset
weights_path = sc.resolve_dataset_file_path(
    "model_weights", owner_email="{CANADA_EMAIL}"
)
with open(weights_path) as f:
    weights = json.load(f)
model = NanoLM(weights)

# Load evaluation prompts from Italy's private dataset
prompt_path = sc.resolve_dataset_file_path(
    "eval_prompts", owner_email="{ITALY_EMAIL}"
)
with open(prompt_path) as f:
    prompts = list(csv.DictReader(f))

# Run inference
results = []
for row in prompts:
    completion = model.generate(row["prompt"])
    results.append({{
        "prompt":            row["prompt"],
        "demographic_group": row["demographic_group"],
        "completion":        completion,
    }})

os.makedirs("outputs", exist_ok=True)
with open("outputs/bias_eval_results.json", "w") as f:
    json.dump({{"total_prompts": len(results), "results": results}}, f, indent=2)

print(f"Inference complete. {{len(results)}} prompts evaluated.")
'''

print(JOB_CODE)

In [ ]:
code_path = create_code_file(JOB_CODE)

researcher.submit_python_job(
    ENCLAVE_EMAIL,
    code_path,
    "bias_eval_job",
    datasets={
        CANADA_EMAIL: ["model_weights"],
        ITALY_EMAIL:  ["eval_prompts"],
    },
    share_results_with_do=True,
)
print(f"  Job 'bias_eval_job' submitted to the enclave ({ENCLAVE_EMAIL})")

## Step 6 — Wait for the enclave to distribute the job

The enclave process syncs the submission and forwards the approval request to
Canada and Italy. We poll until both can see the job.

In [ ]:
JOB_NAME = "bias_eval_job"


def job_distributed():
    canada.sync()
    italy.sync()
    seen_by_canada = any(j.name == JOB_NAME for j in canada.jobs)
    seen_by_italy  = any(j.name == JOB_NAME for j in italy.jobs)
    return seen_by_canada and seen_by_italy


wait_for("the enclave to distribute the job to Canada and Italy", job_distributed)

canada_job = next(j for j in canada.jobs if j.name == JOB_NAME)
italy_job  = next(j for j in italy.jobs if j.name == JOB_NAME)
print(f"  Canada sees '{JOB_NAME}'  status={canada_job.status}")
print(f"  Italy  sees '{JOB_NAME}'  status={italy_job.status}")

## Step 7 — Canada and Italy review and approve

Inspect `canada_job` / `italy_job` above before approving. The enclave proceeds
only once **both** have approved.

In [ ]:
canada.approve_job(next(j for j in canada.jobs if j.name == JOB_NAME))
print("  Canada approved")

italy.approve_job(next(j for j in italy.jobs if j.name == JOB_NAME))
print("  Italy  approved")

## Step 8 — Wait for the enclave to run the job and distribute results

The enclave collects both approvals, runs the inference against the model
weights and prompts, and distributes the outputs. We poll the researcher's job
until it reaches `done`.

In [ ]:
def results_ready():
    researcher.sync()
    jobs = [j for j in researcher.jobs if j.name == JOB_NAME]
    return bool(jobs) and jobs[0].status == "done"


wait_for("the enclave to run the job and distribute results", results_ready, timeout=420)
print("  Enclave finished — results available")

## Step 9 — Researcher retrieves the result

In [ ]:
researcher.sync()
researcher_job = next(j for j in researcher.jobs if j.name == JOB_NAME)
print(f"  Researcher job status : {researcher_job.status}")
print(f"  Output files          : {researcher_job.output_paths}")

assert researcher_job.status == "done", researcher_job.status
assert len(researcher_job.output_paths) > 0

with open(researcher_job.output_paths[0]) as f:
    result = json.load(f)

print()
print(f"  Total prompts evaluated : {result['total_prompts']}")
print()
for r in result["results"]:
    print(f"  [{r['demographic_group']}]")
    print(f"    prompt     : {r['prompt']}")
    print(f"    completion : {r['completion']}")
    print()

print("  Researcher received and validated the inference results")

## Step 10 — Canada and Italy also receive the result

Because `share_results_with_do=True`, each data owner independently receives the
output.

In [ ]:
canada.sync()
italy.sync()

canada_job = next(j for j in canada.jobs if j.name == JOB_NAME)
italy_job  = next(j for j in italy.jobs if j.name == JOB_NAME)

print(f"  Canada — output files : {canada_job.output_paths}")
print(f"  Italy  — output files : {italy_job.output_paths}")

assert len(canada_job.output_paths) > 0
assert len(italy_job.output_paths)  > 0

print()
print("  Both Canada and Italy received the result")